# Manual sequence annotation

Pick a set, view images with number labels, define an ordered sequence, save.

In [ ]:
import json, os
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display as ipy_display
import ipywidgets as widgets
from google.colab import drive

drive.mount('/content/drive')

drive_root    = "/content/drive/MyDrive/Gh0st in the Loop"
prepared_root = f"{drive_root}/dataset/prepared"
outputs_dir   = f"{drive_root}/outputs"

SUPPORTED = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}

# Set list from prepared directory
set_names = sorted(d.name for d in Path(prepared_root).iterdir() if d.is_dir())
print(f"{len(set_names)} sets available")

In [ ]:
# ── Pick a set and view images with labels ────────────────────────────────────
set_dropdown = widgets.Dropdown(options=set_names, description="Set:",
                                layout=widgets.Layout(width='300px'))
out = widgets.Output()

label_map   = {}   # label (int) → full image path (str)
current_set = None

def show_set(change=None):
    global label_map, current_set
    current_set = set_dropdown.value
    folder = Path(prepared_root) / current_set
    paths  = sorted(p for p in folder.rglob("*") if p.suffix.lower() in SUPPORTED)

    label_map = {i + 1: str(p) for i, p in enumerate(paths)}

    thumb, label_h = 180, 28
    cols = min(len(paths), 8)
    rows = (len(paths) + cols - 1) // cols
    grid = Image.new("RGB", (cols * thumb, rows * (thumb + label_h)), (30, 30, 30))
    draw = ImageDraw.Draw(grid)

    for i, (label, path) in enumerate(label_map.items()):
        img = Image.open(path).convert("RGB").resize((thumb, thumb))
        row, col = divmod(i, cols)
        grid.paste(img, (col * thumb, row * (thumb + label_h)))
        draw.text(
            (col * thumb + thumb // 2 - 4, row * (thumb + label_h) + thumb + 6),
            str(label), fill=(255, 220, 0)
        )

    out.clear_output(wait=True)
    with out:
        ipy_display(grid)
        print(f"\n{current_set} — {len(paths)} images")

set_dropdown.observe(show_set, names='value')
ipy_display(set_dropdown, out)
show_set()

In [ ]:
# ── Define the sequence ───────────────────────────────────────────────────────
# Use the numbers shown in the grid above. Repetition is allowed.

sequence_labels = [1, 2, 3]  # ← replace with your ordering

In [ ]:
# ── Preview and save ──────────────────────────────────────────────────────────
print(f"Set     : {current_set}")
print(f"Sequence: {sequence_labels}\n")

missing = [l for l in sequence_labels if l not in label_map]
if missing:
    raise ValueError(f"Unknown labels: {missing} — valid range is 1–{len(label_map)}")

for label in sequence_labels:
    print(f"  {label:>2}  {Path(label_map[label]).name}")

ordered_images = [label_map[l] for l in sequence_labels]

# Load or create manual_sequence.json
manual_path = f"{outputs_dir}/manual_sequence.json"
if os.path.exists(manual_path):
    with open(manual_path) as f:
        manual = json.load(f)
else:
    manual = {"sets": []}

manual["sets"] = [s for s in manual["sets"] if s["name"] != current_set]
manual["sets"].append({"name": current_set, "images": ordered_images})

with open(manual_path, "w") as f:
    json.dump(manual, f, indent=2)

print(f"\n✅ Saved to {manual_path}")
print(f"   Sets with manual ordering: {[s['name'] for s in manual['sets']]}")